In [0]:
create or replace table adb_databricks_careflow.gold.patients_360 as
with cte_lab_result_agg as(select 
patient_id,
count(*) as total_test,
sum(case when result_status = "Abnormal" then 1 else 0 end) as abnormal_test_count,
sum(case when result_status = "Critical" then 1 else 0 end) as critical_test_count,
max(test_date) as last_test_date
from adb_databricks_careflow.silver.lab_results
group by patient_id
)
select
p.patient_id,
p.first_name,
p.last_name,
CAST(p.date_of_birth AS DATE) AS date_of_birth,
p.gender,
p.city,
p.state,
p.insurance_provider_id,
h.hospital_id,
h.hospital_name,
lp.provider_name,
l.total_test,
l.abnormal_test_count,
l.critical_test_count,
l.last_test_date
from adb_databricks_careflow.silver.patients as p
left join adb_databricks_careflow.silver.hospitals as h on p.registered_hospital_id
 = h.hospital_id
left join adb_databricks_careflow.silver.insurance_providers as lp on p.insurance_provider_id = lp.insurance_provider_id
left join cte_lab_result_agg as l on p.patient_id = l.patient_id




In [0]:
%python
updated_df = spark.table("adb_databricks_careflow.gold.patients_360")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))